In [0]:
%run ../includes/storage_config


In [0]:
# load common functions
from utils.common_functions import merge_data

In [0]:
raw_df = spark.table(raw_table_name)

### Step 1 - Filter date and convert data types 

In [0]:
# Step 1 - select only trains from NS
from pyspark.sql.functions import col, to_timestamp, to_date, regexp_replace
ns_df = raw_df.filter(col("service_company") == "NS")

# Step 2 - drop unnecessary columns
ns_df = ns_df.drop("stop_planned_platform", "stop_actual_platform", "stop_platform_change")

# Step 3 - type casting 

df_no_timezone = ns_df.withColumn("stop_arrival_time", regexp_replace("stop_arrival_time", r"[+-]\d{2}:\d{2}$", "")) \
    .withColumn("stop_departure_time", regexp_replace("stop_departure_time", r"[+-]\d{2}:\d{2}$", ""))

df_with_timestamp = df_no_timezone.withColumn("service_date", to_date("service_date") ) \
    .withColumn("stop_arrival_time", to_timestamp("stop_arrival_time", "yyyy-MM-dd'T'HH:mm:ss") ) \
    .withColumn("stop_departure_time", to_timestamp("stop_departure_time", "yyyy-MM-dd'T'HH:mm:ss") )

#display(df_with_timestamp.select('stop_station_name').distinct()) # double check that all stops are within the Netherlands

In [0]:
# display(df_with_timestamp.limit(20))

### Step 2 - split data by stop and services

In [0]:
from pyspark.sql.functions import col

# the stop-service keeps a one-to-many relationship

# stop id as the primary key 
df_stops_ns = df_with_timestamp.select(col("service_date"), col("service_rdt_id"), col("stop_rdt_id"), col("stop_station_code"), \
        col("stop_station_name"), col("stop_arrival_time"), col("stop_arrival_delay"), col("stop_arrival_cancelled"), \
        col("stop_departure_time"), col("stop_departure_delay"), col("stop_departure_cancelled"), col("file_source"), \
        col("ingestion_date"))

# service id as the primary key
df_services_ns = df_with_timestamp.select(col("service_rdt_id"), col("service_date"), col("service_type"), \
        col("service_train_number"), col("service_completely_cancelled"), col("service_partly_cancelled"), \
        col("service_maximum_delay"), col("file_source"), col("ingestion_date")) \
        .dropDuplicates(["service_rdt_id"])


In [0]:
# run the commands below to check the number of rows of the new data frames for data integrity
# df_with_timestamp.count()
# df_services_ns.count()
# df_stops_ns.select('stop_rdt_id').distinct().count()

### Step 3 - write data into processed layer and create schema

In [0]:
stop_table_location = f'{schema}.{processed_table_name_stops}'
services_table_location = f'{schema}.{processed_table_name_services}'
merge_condition_stops = 'source.stop_rdt_id = target.stop_rdt_id'
merge_condition_services = 'source.service_rdt_id = target.service_rdt_id'
# Use merge_data function here
merge_data(
    input_df=df_stops_ns,
    table_name=processed_table_name_stops,
    schema_name = schema,
    merge_condition= merge_condition_stops,
    spark=spark
)
merge_data(
    input_df=df_services_ns,
    table_name=processed_table_name_services,
    schema_name = schema,
    merge_condition= merge_condition_services,
    spark=spark
)
# df_stops_ns.write.mode("overwrite").format("delta").saveAsTable(stop_table_location)
# df_services_ns.write.mode("overwrite").format("delta").saveAsTable(services_table_location)

In [0]:
dbutils.notebook.exit("Success")